#### Load Tree Data

#### Here the two data files in this directory:

- `2015_Street_Tree_Census_-_Tree_Data_20260522.csv`
- `favorite-trees.json`

In [1]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path.cwd()

tree_csv = DATA_DIR / "2015_Street_Tree_Census_-_Tree_Data_20260522.csv"
favorites_json = DATA_DIR / "favorite-trees.json"

tree_csv, favorites_json

(PosixPath('/Users/yanchen/Desktop/Proj/Tree Data/notebooks/2015_Street_Tree_Census_-_Tree_Data_20260522.csv'),
 PosixPath('/Users/yanchen/Desktop/Proj/Tree Data/notebooks/favorite-trees.json'))

In [3]:
# Load the 2015 street tree census CSV.
trees = pd.read_csv(
    tree_csv,
    low_memory=False,
    parse_dates=["created_at"],
)

trees.shape

(683788, 45)

In [3]:
trees.head()

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,boro_ct,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl
0,180683,348711,2015-08-27,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,...,"4,073,900",New York,40.723092,-73.844215,"1,027,431.148","202,756.7687",29.0,739,4052307.0,4.022210e+09
1,200540,315986,2015-09-03,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,...,"4,097,300",New York,40.794111,-73.818679,"1,034,455.701","228,644.8374",19.0,973,4101931.0,4.044750e+09
2,204026,218365,2015-09-05,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,"3,044,900",New York,40.717581,-73.936608,"1,001,822.831","200,716.8913",34.0,449,3338310.0,3.028870e+09
3,204337,217969,2015-09-05,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,"3,044,900",New York,40.713537,-73.934456,"1,002,420.358","199,244.2531",34.0,449,3338342.0,3.029250e+09
4,189565,223043,2015-08-30,21,0,OnCurb,Alive,Good,Tilia americana,American linden,...,"3,016,500",New York,40.666778,-73.975979,"990,913.775","182,202.426",39.0,165,3025654.0,3.010850e+09


In [5]:
# Load and normalize the favorite trees JSON.
with favorites_json.open("r", encoding="utf-8") as f:
    favorite_payload = json.load(f)

favorites = pd.DataFrame(favorite_payload["data"]).rename(
    columns={
        "treeId": "tree_id",
        "numberOfTimesFavorited": "number_of_times_favorited",
    }
)

favorites.shape

(5171, 2)

In [6]:
favorites.head()

,tree_id,number_of_times_favorited
0,90423,1
1,104829,1
2,108832,1
3,118055,1
4,136058,1


In [7]:
favorites.info()

<class 'pandas.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 2 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   tree_id                    5171 non-null   int64
 1   number_of_times_favorited  5171 non-null   int64
dtypes: int64(2)
memory usage: 80.9 KB


In [8]:
census_tree_ids = set(pd.to_numeric(trees["tree_id"], errors="coerce").dropna().astype("int64"))
favorite_tree_ids = set(pd.to_numeric(favorites["tree_id"], errors="coerce").dropna().astype("int64"))
matching_tree_ids = census_tree_ids & favorite_tree_ids

pd.DataFrame({
    "metric": [
        "Unique tree IDs in census data",
        "Unique tree IDs in favorites data",
        "Unique tree IDs in both datasets",
    ],
    "count": [
        len(census_tree_ids),
        len(favorite_tree_ids),
        len(matching_tree_ids),
    ],
})

,metric,count
0,Unique tree IDs in census data,683788
1,Unique tree IDs in favorites data,5171
2,Unique tree IDs in both datasets,425


In [9]:
trees["tree_id"] = pd.to_numeric(trees["tree_id"], errors="coerce").astype("Int64")
favorites["tree_id"] = pd.to_numeric(favorites["tree_id"], errors="coerce").astype("Int64")

trees_with_favorites = trees.merge(
    favorites[["tree_id"]],
    on="tree_id",
    how="left",
    indicator=True
)

trees_with_favorites["favorite"] = (
    trees_with_favorites["_merge"].eq("both").astype(int)
)



In [10]:
trees_with_favorites.head()

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,_merge,favorite
0,180683,348711,2015-08-27,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,...,40.723092,-73.844215,"1,027,431.148","202,756.7687",29.0,739,4052307.0,4.022210e+09,left_only,0
1,200540,315986,2015-09-03,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,...,40.794111,-73.818679,"1,034,455.701","228,644.8374",19.0,973,4101931.0,4.044750e+09,left_only,0
2,204026,218365,2015-09-05,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,40.717581,-73.936608,"1,001,822.831","200,716.8913",34.0,449,3338310.0,3.028870e+09,left_only,0
3,204337,217969,2015-09-05,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,40.713537,-73.934456,"1,002,420.358","199,244.2531",34.0,449,3338342.0,3.029250e+09,left_only,0
4,189565,223043,2015-08-30,21,0,OnCurb,Alive,Good,Tilia americana,American linden,...,40.666778,-73.975979,"990,913.775","182,202.426",39.0,165,3025654.0,3.010850e+09,left_only,0


In [11]:
trees_with_favorites[trees_with_favorites['favorite']==1]

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,_merge,favorite
1006,190164,345815,2015-08-30,13,0,OnCurb,Alive,Poor,Acer platanoides,Norway maple,...,40.711972,-73.906723,"1,010,109.289","198,680.6946",30.0,595,4433970.0,4.033660e+09,both,1
2704,201896,228570,2015-09-04,14,0,OnCurb,Alive,Good,Pyrus calleryana,Callery pear,...,40.635801,-73.980435,"989,680.1896","170,916.3026",39.0,478,3126513.0,3.053820e+09,both,1
6248,212398,108911,2015-09-09,18,0,OffsetFromCurb,Alive,Good,Quercus palustris,pin oak,...,40.832013,-73.946373,"999,090.3138","242,406.5047",7.0,237,1084180.0,1.020850e+09,both,1
7501,202475,407173,2015-09-04,2,0,OnCurb,Alive,Good,Ginkgo biloba,ginkgo,...,40.593487,-74.160143,"939,774.6014","155,540.5446",51.0,"27,706",5037497.0,5.023620e+09,both,1
9411,189423,312651,2015-08-30,23,0,OnCurb,Alive,Good,Tilia cordata,littleleaf linden,...,40.765506,-73.796739,"1,040,554.893","218,236.6522",19.0,"1,141",4118871.0,4.052530e+09,both,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667787,156934,217030,2015-08-18,10,0,OnCurb,Alive,Good,Tilia americana,American linden,...,40.706009,-73.961342,"994,968.2","196,497.1205",33.0,535,3059945.0,3.021820e+09,both,1
670619,191777,224066,2015-08-31,23,0,OnCurb,Alive,Good,Quercus palustris,pin oak,...,40.680979,-74.000854,"984,013.1123","187,375.5938",39.0,65,3005082.0,3.003600e+09,both,1
671019,167344,228798,2015-08-22,17,0,OnCurb,Alive,Good,Fraxinus pennsylvanica,green ash,...,40.638702,-73.981702,"989,328.2133","171,973.1102",39.0,228,3124992.0,3.053478e+09,both,1
674378,192605,103302,2015-08-31,13,0,OffsetFromCurb,Alive,Good,Platanus x acerifolia,London planetree,...,40.718632,-73.990672,"986,835.6995","201,093.6632",1.0,18,1005462.0,1.004140e+09,both,1


In [12]:
trees_with_favorites = trees_with_favorites.drop(columns="_merge")

trees_with_favorites.head()

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
0,180683,348711,2015-08-27,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,...,New York,40.723092,-73.844215,"1,027,431.148","202,756.7687",29.0,739,4052307.0,4.022210e+09,0
1,200540,315986,2015-09-03,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,...,New York,40.794111,-73.818679,"1,034,455.701","228,644.8374",19.0,973,4101931.0,4.044750e+09,0
2,204026,218365,2015-09-05,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.717581,-73.936608,"1,001,822.831","200,716.8913",34.0,449,3338310.0,3.028870e+09,0
3,204337,217969,2015-09-05,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.713537,-73.934456,"1,002,420.358","199,244.2531",34.0,449,3338342.0,3.029250e+09,0
4,189565,223043,2015-08-30,21,0,OnCurb,Alive,Good,Tilia americana,American linden,...,New York,40.666778,-73.975979,"990,913.775","182,202.426",39.0,165,3025654.0,3.010850e+09,0


In [23]:
trees_with_favorites_live = trees_with_favorites[

    ~trees_with_favorites["status"].isin(["Dead"])

].copy()
trees_with_favorites_live


,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
0,180683,348711,2015-08-27,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,...,New York,40.723092,-73.844215,"1,027,431.148","202,756.7687",29.0,739,4052307.0,4.022210e+09,0
1,200540,315986,2015-09-03,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,...,New York,40.794111,-73.818679,"1,034,455.701","228,644.8374",19.0,973,4101931.0,4.044750e+09,0
2,204026,218365,2015-09-05,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.717581,-73.936608,"1,001,822.831","200,716.8913",34.0,449,3338310.0,3.028870e+09,0
3,204337,217969,2015-09-05,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.713537,-73.934456,"1,002,420.358","199,244.2531",34.0,449,3338342.0,3.029250e+09,0
4,189565,223043,2015-08-30,21,0,OnCurb,Alive,Good,Tilia americana,American linden,...,New York,40.666778,-73.975979,"990,913.775","182,202.426",39.0,165,3025654.0,3.010850e+09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683783,155433,217978,2015-08-18,25,0,OnCurb,Alive,Good,Quercus palustris,pin oak,...,New York,40.713211,-73.954944,"996,740.686","199,121.6363",34.0,519,3062513.0,3.023690e+09,0
683784,183795,348185,2015-08-29,7,0,OnCurb,Alive,Good,Cladrastis kentukea,Kentucky yellowwood,...,New York,40.715194,-73.856650,"1,023,989.074","199,873.6475",29.0,707,4075448.0,4.031810e+09,0
683785,166161,401670,2015-08-22,12,0,OnCurb,Alive,Good,Acer rubrum,red maple,...,New York,40.620762,-74.136517,"946,351.4104","165,466.0763",50.0,201,5011657.0,5.004080e+09,0
683786,184028,504204,2015-08-29,9,0,OnCurb,Alive,Good,Acer rubrum,red maple,...,New York,40.850828,-73.903115,"1,011,053.646","249,271.9507",15.0,"23,502",2007757.0,2.028120e+09,0


In [24]:
max_missing_fraction = 0.03
trees_with_favorites_live = trees_with_favorites_live[
    trees_with_favorites_live.isna().mean(axis=1) <= max_missing_fraction
].copy()

In [25]:
trees_with_favorites_live

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
5,190422,106099,2015-08-30,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.770046,-73.984950,"988,418.6997","219,825.5227",3.0,145,1076229.0,1.011310e+09,0
6,190426,106099,2015-08-30,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.770210,-73.985338,"988,311.19","219,885.2785",3.0,145,1076229.0,1.011310e+09,0
13,189465,219493,2015-08-30,22,0,OnCurb,Alive,Good,Platanus x acerifolia,London planetree,...,New York,40.694733,-73.968211,"993,065.3039","192,388.0651",35.0,191,3054331.0,3.018880e+09,0
14,192998,211160,2015-08-31,30,0,OnCurb,Alive,Fair,Platanus x acerifolia,London planetree,...,New York,40.664317,-73.921130,"1,006,130.777","181,314.9855",41.0,900,3081177.0,3.035310e+09,0
15,189834,219505,2015-08-30,12,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.693314,-73.967601,"993,234.6165","191,871.1586",35.0,191,3054408.0,3.018890e+09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683755,189595,223272,2015-08-30,20,0,OnCurb,Alive,Good,Ginkgo biloba,ginkgo,...,New York,40.671685,-73.975309,"991,099.1352","183,990.1468",39.0,157,3325195.0,3.010710e+09,0
683756,208765,107164,2015-09-08,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.763224,-73.960984,"995,058.2162","217,342.1992",5.0,118,1044692.0,1.014390e+09,0
683768,190040,106508,2015-08-30,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.786150,-73.971152,"992,238.5526","225,693.6953",6.0,173,1031403.0,1.012000e+09,0
683772,189461,219493,2015-08-30,4,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.695204,-73.968306,"993,038.7816","192,559.4246",35.0,191,3054324.0,3.018880e+09,0


In [26]:
trees_with_favorites_dead=trees_with_favorites[

    trees_with_favorites["status"].isin(["Dead"])

].copy()

In [27]:
trees_with_favorites_dead

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
57,187807,506266,2015-08-29,0,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.871927,-73.882349,"1,016,788.348","256,965.9677",11.0,415,2016979.0,2.032990e+09,0
196,208322,222858,2015-09-07,10,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.705742,-73.917849,"1,007,026.93","196,407.8326",37.0,445,3073739.0,3.032390e+09,0
209,209058,415850,2015-09-08,6,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.559061,-74.106038,"954,785.6764","142,975.3788",50.0,"12,806",5056700.0,5.040740e+09,0
266,210544,216081,2015-09-08,6,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.726271,-73.947287,"998,860.6187","203,881.213",33.0,571,3066265.0,3.026510e+09,0
285,188609,107627,2015-08-30,9,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.771992,-73.951657,"997,639.9876","220,537.9707",5.0,136,1050348.0,1.015590e+09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682816,184342,505510,2015-08-29,11,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.855136,-73.892756,"1,013,917.462","250,844.6335",15.0,385,2092226.0,2.030530e+09,0
682823,155430,217978,2015-08-18,2,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.713867,-73.954529,"996,855.5617","199,360.6646",33.0,519,3062288.0,3.023390e+09,0
683346,156566,514686,2015-08-18,10,0,OnCurb,Dead,NaN,NaN,NaN,...,New York,40.889713,-73.851001,"1,025,446.985","263,459.2887",12.0,424,2064704.0,2.048660e+09,0
683693,156162,108193,2015-08-18,3,0,OffsetFromCurb,Dead,NaN,NaN,NaN,...,New York,40.785136,-73.951109,"997,789.3617","225,326.7807",5.0,"15,801",1048443.0,1.015230e+09,0


In [28]:
trees_with_favorites_stump=trees_with_favorites[

    trees_with_favorites["status"].isin(["Stump"])

].copy()

trees_with_favorites_stump

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
37,211205,302080,2015-09-09,0,16,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.774993,-73.922037,"1,005,843.393","221,637.1447",22.0,95,4019192.0,4.008740e+09,0
239,203597,301947,2015-09-04,0,15,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.772483,-73.909073,"1,009,434.936","220,726.1774",22.0,117,4015864.0,4.008060e+09,0
641,179766,230466,2015-08-27,0,30,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.637379,-73.953814,"997,068.6473","171,494.1029",45.0,770,3120745.0,3.052240e+09,0
644,180619,348834,2015-08-27,0,10,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.725300,-73.839376,"1,028,771.091","203,563.63",29.0,"75,702",4052666.0,4.022430e+09,0
646,208828,107238,2015-09-08,0,15,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.764914,-73.960857,"995,092.9675","217,957.8614",5.0,118,1044738.0,1.014418e+09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683348,152836,217718,2015-08-17,0,13,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.711246,-73.956418,"996,332.3328","198,405.4546",34.0,523,3063024.0,3.024090e+09,0
683686,191219,321689,2015-08-31,0,25,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.761032,-73.765688,"1,049,160.63","216,628.0173",19.0,"1,471",4138506.0,4.063100e+09,0
683695,170888,231931,2015-08-24,0,5,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.647154,-73.955619,"996,565.8182","175,055.125",40.0,794,3117748.0,3.051270e+09,0
683697,177922,410634,2015-08-26,0,17,OnCurb,Stump,NaN,NaN,NaN,...,New York,40.528544,-74.165246,"938,312.5062","131,882.712",51.0,176,5081744.0,5.063570e+09,0


In [30]:
import plotly.express as px

# Make sure latitude and longitude are numeric

trees_with_favorites_live["latitude"] = trees_with_favorites_live["latitude"].astype(float)

trees_with_favorites_live["longitude"] = trees_with_favorites_live["longitude"].astype(float)

# Remove rows with missing coordinates

trees_map_df = trees_with_favorites_live.dropna(subset=["latitude", "longitude"]).copy()

fig = px.scatter_map(

    trees_map_df,

    lat="latitude",

    lon="longitude",

    hover_name="spc_common",   # tree common name


    zoom=10,

    height=700

)

fig.update_layout(

    mapbox_style="open-street-map",

    margin={"r":0, "t":0, "l":0, "b":0}

)

fig.show()

In [31]:
import plotly.express as px

# Make sure latitude and longitude are numeric

trees_with_favorites_dead["latitude"] = trees_with_favorites_dead["latitude"].astype(float)

trees_with_favorites_dead["longitude"] = trees_with_favorites_dead["longitude"].astype(float)

# Remove rows with missing coordinates

trees_map_df2 = trees_with_favorites_dead.dropna(subset=["latitude", "longitude"]).copy()

fig = px.scatter_map(

    trees_map_df2,

    lat="latitude",

    lon="longitude",

    hover_name="spc_common",   # tree common name


    zoom=10,

    height=700

)

fig.update_layout(

    mapbox_style="open-street-map",

    margin={"r":0, "t":0, "l":0, "b":0}

)

fig.show()

In [32]:
trees_with_favorites_live

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,...,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,favorite
5,190422,106099,2015-08-30,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.770046,-73.984950,"988,418.6997","219,825.5227",3.0,145,1076229.0,1.011310e+09,0
6,190426,106099,2015-08-30,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.770210,-73.985338,"988,311.19","219,885.2785",3.0,145,1076229.0,1.011310e+09,0
13,189465,219493,2015-08-30,22,0,OnCurb,Alive,Good,Platanus x acerifolia,London planetree,...,New York,40.694733,-73.968211,"993,065.3039","192,388.0651",35.0,191,3054331.0,3.018880e+09,0
14,192998,211160,2015-08-31,30,0,OnCurb,Alive,Fair,Platanus x acerifolia,London planetree,...,New York,40.664317,-73.921130,"1,006,130.777","181,314.9855",41.0,900,3081177.0,3.035310e+09,0
15,189834,219505,2015-08-30,12,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.693314,-73.967601,"993,234.6165","191,871.1586",35.0,191,3054408.0,3.018890e+09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683755,189595,223272,2015-08-30,20,0,OnCurb,Alive,Good,Ginkgo biloba,ginkgo,...,New York,40.671685,-73.975309,"991,099.1352","183,990.1468",39.0,157,3325195.0,3.010710e+09,0
683756,208765,107164,2015-09-08,11,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.763224,-73.960984,"995,058.2162","217,342.1992",5.0,118,1044692.0,1.014390e+09,0
683768,190040,106508,2015-08-30,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.786150,-73.971152,"992,238.5526","225,693.6953",6.0,173,1031403.0,1.012000e+09,0
683772,189461,219493,2015-08-30,4,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,...,New York,40.695204,-73.968306,"993,038.7816","192,559.4246",35.0,191,3054324.0,3.018880e+09,0


In [33]:
trees_with_favorites_live = trees_with_favorites_live.drop(columns=["block_id", "created_at", "community board", "borocode",
                                                                    "borough",	"cncldist","st_assem","st_senate",	"nta", "x_sp","y_sp",	
                                                                    "council district",	"census tract",	"bin",	"bbl", "stump_diam",'boro_ct'])




In [34]:
trees_with_favorites_live

,tree_id,tree_dbh,curb_loc,status,health,spc_latin,spc_common,steward,guards,sidewalk,...,brch_shoe,brch_other,address,postcode,zip_city,nta_name,state,latitude,longitude,favorite
5,190422,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,8 COLUMBUS AVENUE,10023,New York,Lincoln Square,New York,40.770046,-73.984950,0
6,190426,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,120 WEST 60 STREET,10023,New York,Lincoln Square,New York,40.770210,-73.985338,0
13,189465,22,OnCurb,Alive,Good,Platanus x acerifolia,London planetree,3or4,Harmful,NoDamage,...,No,No,100 WAVERLY AVENUE,11205,Brooklyn,Clinton Hill,New York,40.694733,-73.968211,0
14,192998,30,OnCurb,Alive,Fair,Platanus x acerifolia,London planetree,1or2,NaN,Damage,...,No,Yes,2126 UNION STREET,11212,Brooklyn,Brownsville,New York,40.664317,-73.921130,0
15,189834,12,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,3or4,Helpful,NoDamage,...,No,No,449 MYRTLE AVENUE,11205,Brooklyn,Clinton Hill,New York,40.693314,-73.967601,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683755,189595,20,OnCurb,Alive,Good,Ginkgo biloba,ginkgo,1or2,Helpful,Damage,...,No,No,31 FISKE PLACE,11215,Brooklyn,Park Slope-Gowanus,New York,40.671685,-73.975309,0
683756,208765,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Harmful,NoDamage,...,No,No,325 EAST 64 STREET,10065,New York,Lenox Hill-Roosevelt Island,New York,40.763224,-73.960984,0
683768,190040,10,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Harmful,NoDamage,...,No,No,41 WEST 86 STREET,10024,New York,Upper West Side,New York,40.786150,-73.971152,0
683772,189461,4,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,86 WAVERLY AVENUE,11205,Brooklyn,Clinton Hill,New York,40.695204,-73.968306,0


In [35]:
trees_with_favorites_dead= trees_with_favorites_dead.drop(columns=["block_id", "created_at", "community board", "borocode",
                                                                    "borough",	"cncldist","st_assem","st_senate",	"nta", "x_sp","y_sp",	
                                                                    "council district",	"census tract",	"bin",	"bbl", "stump_diam",'boro_ct'])
trees_with_favorites_dead = trees_with_favorites_dead.dropna(axis=1)

In [36]:
trees_with_favorites_dead

,tree_id,tree_dbh,curb_loc,status,user_type,root_stone,root_grate,root_other,trunk_wire,trnk_light,...,brch_shoe,brch_other,address,postcode,zip_city,nta_name,state,latitude,longitude,favorite
57,187807,0,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,300 EAST MOSHOLU PARKWAY SOUTH,10458,Bronx,Norwood,New York,40.871927,-73.882349,0
196,208322,10,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,1691 DEKALB AVENUE,11237,Brooklyn,Bushwick North,New York,40.705742,-73.917849,0
209,209058,6,OnCurb,Dead,TreesCount Staff,No,No,No,No,No,...,No,No,295 ROMA AVENUE,10306,Staten Island,Oakwood-Oakwood Beach,New York,40.559061,-74.106038,0
266,210544,6,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,111 DIAMOND STREET,11222,Brooklyn,Greenpoint,New York,40.726271,-73.947287,0
285,188609,9,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,413 EAST 79 STREET,10075,New York,Yorkville,New York,40.771992,-73.951657,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682816,184342,11,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,489 EAST 183 STREET,10458,Bronx,Claremont-Bathgate,New York,40.855136,-73.892756,0
682823,155430,2,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,417 METROPOLITAN AVENUE,11211,Brooklyn,North Side-South Side,New York,40.713867,-73.954529,0
683346,156566,10,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,936 EAST 231 STREET,10466,Bronx,Williamsbridge-Olinville,New York,40.889713,-73.851001,0
683693,156162,3,OffsetFromCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,156 EAST 95 STREET,10128,New York,Upper East Side-Carnegie Hill,New York,40.785136,-73.951109,0


In [37]:
trees_with_favorites_stump= trees_with_favorites_stump.drop(columns=["block_id", "created_at", "community board", "borocode",
                                                                    "borough",	"cncldist","st_assem","st_senate",	"nta", "x_sp","y_sp",	
                                                                    "council district",	"census tract",	"bin",	"bbl", "tree_dbh",'boro_ct'])
trees_with_favorites_stump = trees_with_favorites_stump.dropna(axis=1)

In [38]:
trees_with_favorites_stump

,tree_id,stump_diam,curb_loc,status,user_type,root_stone,root_grate,root_other,trunk_wire,trnk_light,...,brch_shoe,brch_other,address,postcode,zip_city,nta_name,state,latitude,longitude,favorite
37,211205,16,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,21-023 24 DRIVE,11102,Astoria,Steinway,New York,40.774993,-73.922037,0
239,203597,15,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,22-063 37 STREET,11105,Astoria,Steinway,New York,40.772483,-73.909073,0
641,179766,30,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,2676 BEDFORD AVENUE,11210,Brooklyn,Flatbush,New York,40.637379,-73.953814,0
644,180619,10,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,113-001 JEWEL AVENUE,11375,Forest Hills,Forest Hills,New York,40.725300,-73.839376,0
646,208828,15,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,301 EAST 66 STREET,10065,New York,Lenox Hill-Roosevelt Island,New York,40.764914,-73.960857,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683348,152836,13,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,281 SOUTH 2 STREET,11211,Brooklyn,North Side-South Side,New York,40.711246,-73.956418,0
683686,191219,25,OnCurb,Stump,NYC Parks Staff,No,No,No,No,No,...,No,No,43-035 216 STREET,11361,Bayside,Bayside-Bayside Hills,New York,40.761032,-73.765688,0
683695,170888,5,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,2327 BEDFORD AVENUE,11226,Brooklyn,Erasmus,New York,40.647154,-73.955619,0
683697,177922,17,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,419 HOLDRIDGE AVENUE,10312,Staten Island,Annadale-Huguenot-Prince's Bay-Eltingville,New York,40.528544,-74.165246,0


In [39]:
trees_with_favorites_live.columns

Index(['tree_id', 'tree_dbh', 'curb_loc', 'status', 'health', 'spc_latin',
       'spc_common', 'steward', 'guards', 'sidewalk', 'user_type', 'problems',
       'root_stone', 'root_grate', 'root_other', 'trunk_wire', 'trnk_light',
       'trnk_other', 'brch_light', 'brch_shoe', 'brch_other', 'address',
       'postcode', 'zip_city', 'nta_name', 'state', 'latitude', 'longitude',
       'favorite'],
      dtype='str')

In [40]:
trees_with_favorites_live = trees_with_favorites_live.rename(

    columns={
        "tree_id": "tree_id",

    "tree_dbh": "tree_diameter",

    "curb_loc": "curb_location",

    "status": "tree_status",

    "health": "tree_health",

    "spc_latin": "scientific_name",

    "spc_common": "common_name",

    "steward": "stewardship_signs",

    "guards": "tree_guard",

    "sidewalk": "sidewalk_condition",

    "user_type": "data_collector_type",

    "problems": "tree_problems",

    "root_stone": "root_problem_paving_stones",

    "root_grate": "root_problem_metal_grates",

    "root_other": "root_problem_other",

    "trunk_wire": "trunk_problem_wires",

    "trnk_light": "trunk_problem_lights",

    "trnk_other": "trunk_problem_other",

    "brch_light": "branch_problem_lights_wires",

    "brch_shoe": "branch_problem_shoes",

    "brch_other": "branch_problem_other",

    "address": "tree_address",

    "postcode": "postcode",

    "zip_city": "city",

    "nta_name": "neighborhood",

    "boro_ct": "census_tract",

    "state": "state",

    "latitude": "latitude",

    "longitude": "longitude",

    "favorite": "favorite"

}


)

In [41]:
trees_with_favorites_live

,tree_id,tree_diameter,curb_location,tree_status,tree_health,scientific_name,common_name,stewardship_signs,tree_guard,sidewalk_condition,...,branch_problem_shoes,branch_problem_other,tree_address,postcode,city,neighborhood,state,latitude,longitude,favorite
5,190422,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,8 COLUMBUS AVENUE,10023,New York,Lincoln Square,New York,40.770046,-73.984950,0
6,190426,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,120 WEST 60 STREET,10023,New York,Lincoln Square,New York,40.770210,-73.985338,0
13,189465,22,OnCurb,Alive,Good,Platanus x acerifolia,London planetree,3or4,Harmful,NoDamage,...,No,No,100 WAVERLY AVENUE,11205,Brooklyn,Clinton Hill,New York,40.694733,-73.968211,0
14,192998,30,OnCurb,Alive,Fair,Platanus x acerifolia,London planetree,1or2,NaN,Damage,...,No,Yes,2126 UNION STREET,11212,Brooklyn,Brownsville,New York,40.664317,-73.921130,0
15,189834,12,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,3or4,Helpful,NoDamage,...,No,No,449 MYRTLE AVENUE,11205,Brooklyn,Clinton Hill,New York,40.693314,-73.967601,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683755,189595,20,OnCurb,Alive,Good,Ginkgo biloba,ginkgo,1or2,Helpful,Damage,...,No,No,31 FISKE PLACE,11215,Brooklyn,Park Slope-Gowanus,New York,40.671685,-73.975309,0
683756,208765,11,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Harmful,NoDamage,...,No,No,325 EAST 64 STREET,10065,New York,Lenox Hill-Roosevelt Island,New York,40.763224,-73.960984,0
683768,190040,10,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Harmful,NoDamage,...,No,No,41 WEST 86 STREET,10024,New York,Upper West Side,New York,40.786150,-73.971152,0
683772,189461,4,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,Helpful,NoDamage,...,No,No,86 WAVERLY AVENUE,11205,Brooklyn,Clinton Hill,New York,40.695204,-73.968306,0


In [42]:
trees_with_favorites_dead.columns

Index(['tree_id', 'tree_dbh', 'curb_loc', 'status', 'user_type', 'root_stone',
       'root_grate', 'root_other', 'trunk_wire', 'trnk_light', 'trnk_other',
       'brch_light', 'brch_shoe', 'brch_other', 'address', 'postcode',
       'zip_city', 'nta_name', 'state', 'latitude', 'longitude', 'favorite'],
      dtype='str')

In [30]:
trees_with_favorites_dead = trees_with_favorites_dead.rename(columns={
    "tree_id": "tree_id",

    "tree_dbh": "tree_diameter",

    "curb_loc": "curb_location",

    "status": "tree_status",

    "user_type": "data_collector_type",

    "root_stone": "root_problem_paving_stones",

    "root_grate": "root_problem_metal_grates",

    "root_other": "root_problem_other",

    "trunk_wire": "trunk_problem_wires",

    "trnk_light": "trunk_problem_lights",

    "trnk_other": "trunk_problem_other",

    "brch_light": "branch_problem_lights_wires",

    "brch_shoe": "branch_problem_shoes",

    "brch_other": "branch_problem_other",

    "address": "tree_address",

    "postcode": "postcode",

    "zip_city": "city",

    "nta_name": "neighborhood",

    "state": "state",

    "latitude": "latitude",

    "longitude": "longitude",

    "favorite": "favorite"
})

In [43]:
trees_with_favorites_dead

,tree_id,tree_dbh,curb_loc,status,user_type,root_stone,root_grate,root_other,trunk_wire,trnk_light,...,brch_shoe,brch_other,address,postcode,zip_city,nta_name,state,latitude,longitude,favorite
57,187807,0,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,300 EAST MOSHOLU PARKWAY SOUTH,10458,Bronx,Norwood,New York,40.871927,-73.882349,0
196,208322,10,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,1691 DEKALB AVENUE,11237,Brooklyn,Bushwick North,New York,40.705742,-73.917849,0
209,209058,6,OnCurb,Dead,TreesCount Staff,No,No,No,No,No,...,No,No,295 ROMA AVENUE,10306,Staten Island,Oakwood-Oakwood Beach,New York,40.559061,-74.106038,0
266,210544,6,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,111 DIAMOND STREET,11222,Brooklyn,Greenpoint,New York,40.726271,-73.947287,0
285,188609,9,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,413 EAST 79 STREET,10075,New York,Yorkville,New York,40.771992,-73.951657,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682816,184342,11,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,489 EAST 183 STREET,10458,Bronx,Claremont-Bathgate,New York,40.855136,-73.892756,0
682823,155430,2,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,417 METROPOLITAN AVENUE,11211,Brooklyn,North Side-South Side,New York,40.713867,-73.954529,0
683346,156566,10,OnCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,936 EAST 231 STREET,10466,Bronx,Williamsbridge-Olinville,New York,40.889713,-73.851001,0
683693,156162,3,OffsetFromCurb,Dead,Volunteer,No,No,No,No,No,...,No,No,156 EAST 95 STREET,10128,New York,Upper East Side-Carnegie Hill,New York,40.785136,-73.951109,0


In [44]:
trees_with_favorites_stump.columns

Index(['tree_id', 'stump_diam', 'curb_loc', 'status', 'user_type',
       'root_stone', 'root_grate', 'root_other', 'trunk_wire', 'trnk_light',
       'trnk_other', 'brch_light', 'brch_shoe', 'brch_other', 'address',
       'postcode', 'zip_city', 'nta_name', 'state', 'latitude', 'longitude',
       'favorite'],
      dtype='str')

In [45]:
trees_with_favorites_stump = trees_with_favorites_stump.rename(columns={
    "tree_id": "tree_id",

    "stump_diam": "stump_diameter",

    "curb_loc": "curb_location",

    "status": "tree_status",

    "user_type": "data_collector_type",

    "root_stone": "root_problem_paving_stones",

    "root_grate": "root_problem_metal_grates",

    "root_other": "root_problem_other",

    "trunk_wire": "trunk_problem_wires",

    "trnk_light": "trunk_problem_lights",

    "trnk_other": "trunk_problem_other",

    "brch_light": "branch_problem_lights_wires",

    "brch_shoe": "branch_problem_shoes",

    "brch_other": "branch_problem_other",

    "address": "tree_address",

    "postcode": "postcode",

    "zip_city": "city",

    "nta_name": "neighborhood",

    "state": "state",

    "latitude": "latitude",

    "longitude": "longitude",

    "favorite": "favorite"


})

In [46]:
trees_with_favorites_stump

,tree_id,stump_diameter,curb_location,tree_status,data_collector_type,root_problem_paving_stones,root_problem_metal_grates,root_problem_other,trunk_problem_wires,trunk_problem_lights,...,branch_problem_shoes,branch_problem_other,tree_address,postcode,city,neighborhood,state,latitude,longitude,favorite
37,211205,16,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,21-023 24 DRIVE,11102,Astoria,Steinway,New York,40.774993,-73.922037,0
239,203597,15,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,22-063 37 STREET,11105,Astoria,Steinway,New York,40.772483,-73.909073,0
641,179766,30,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,2676 BEDFORD AVENUE,11210,Brooklyn,Flatbush,New York,40.637379,-73.953814,0
644,180619,10,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,113-001 JEWEL AVENUE,11375,Forest Hills,Forest Hills,New York,40.725300,-73.839376,0
646,208828,15,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,301 EAST 66 STREET,10065,New York,Lenox Hill-Roosevelt Island,New York,40.764914,-73.960857,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683348,152836,13,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,281 SOUTH 2 STREET,11211,Brooklyn,North Side-South Side,New York,40.711246,-73.956418,0
683686,191219,25,OnCurb,Stump,NYC Parks Staff,No,No,No,No,No,...,No,No,43-035 216 STREET,11361,Bayside,Bayside-Bayside Hills,New York,40.761032,-73.765688,0
683695,170888,5,OnCurb,Stump,TreesCount Staff,No,No,No,No,No,...,No,No,2327 BEDFORD AVENUE,11226,Brooklyn,Erasmus,New York,40.647154,-73.955619,0
683697,177922,17,OnCurb,Stump,Volunteer,No,No,No,No,No,...,No,No,419 HOLDRIDGE AVENUE,10312,Staten Island,Annadale-Huguenot-Prince's Bay-Eltingville,New York,40.528544,-74.165246,0


In [35]:
trees_with_favorites_stump.to_csv("trees_with_favorites_stump.csv", index=False)

In [36]:
trees_with_favorites_dead.to_csv("trees_with_favorites_dead.csv", index=False)

In [47]:
trees_with_favorites_live.to_csv("trees_with_favorites_live2.csv", index=False)